# Future/Past Brain Activity Videos

This notebook creates glass brain videos showing temporal evolution of future and past predictive activity:
1. Load data for comprehension and production
2. Threshold electrodes at 0.1 joint performance
3. Extract lag values from -1000ms to +1000ms (50ms steps)
4. Generate 4 videos:
   - Comprehension Future (green colormap)
   - Comprehension Past (red colormap)
   - Production Future (green colormap)
   - Production Past (red colormap)

In [7]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import LinearSegmentedColormap
import pandas as pd
import numpy as np
import sys
sys.path.append('/scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/scripts')
import tfsplt_future_past_utils as pu
from nilearn import plotting
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

In [15]:
# Configuration
res_d = "/scratch/gpfs/HASSON/ij9216/projects/code/247/247-encoding-dev/results/tfs"
output_dir = '/scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/videos'
thresh_joint = 0.1

# Lag range for videos: -1000ms to +1000ms in 50ms steps
lag_start = -1000
lag_end = 1000
lag_step = 50
lags_to_plot = np.arange(lag_start, lag_end + lag_step, lag_step)

print(f"Results directory: {res_d}")
print(f"Output directory: {output_dir}")
print(f"Joint threshold: {thresh_joint}")
print(f"Lag range: {lag_start} to {lag_end} ms")
print(f"Number of frames: {len(lags_to_plot)}")

Results directory: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-encoding-dev/results/tfs
Output directory: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/videos
Joint threshold: 0.1
Lag range: -1000 to 1000 ms
Number of frames: 41


## Load Data

In [9]:
# Load comprehension data
print("Loading comprehension data...")
comp_data = pu.load_res_add_roi_threshold(
    f"{res_d}/ij-tfs-%s-gpt2-xl-bandedRidge-lag60-50-all-static_future_past-reph-translate-control_pca300_drop-short_mistral_comp.csv",
    None
)

# Rename lag columns to numeric values
lag_cols = np.arange(-60000, 60001, 50)
n_lag_cols = len(lag_cols)
new_columns = list(lag_cols) + list(comp_data.columns[n_lag_cols:])
comp_data.columns = new_columns

print(f"Comprehension shape: {comp_data.shape}")


Loading comprehension data...
Comprehension shape: (2644, 2406)


In [10]:
# Load production data
print("Loading production data...")
prod_data = pu.load_res_add_roi_threshold(
    f"{res_d}/ij-tfs-%s-gpt2-xl-bandedRidge-lag60-50-all-static_future_past-reph-translate-control_pca300_drop-short_mistral_prod.csv",
    None
)

# Rename lag columns to numeric values
new_columns = list(lag_cols) + list(prod_data.columns[n_lag_cols:])
prod_data.columns = new_columns

print(f"Production shape: {prod_data.shape}")


Loading production data...
Production shape: (2644, 2406)


## Threshold Electrodes and Prepare Data

In [11]:
# Get max joint performance for thresholding
def get_max_joint_for_threshold(df, thresh, thresh_lags=None):
    """Get electrodes with max joint performance above threshold."""
    joint_data = df[df['label3'] == 'joint'].copy()
    
    # Get max correlation across all lags for each electrode
    lag_columns = [col for col in joint_data.columns if isinstance(col, (int, float, np.integer, np.floating))]
    joint_data['max_joint'] = joint_data[lag_columns].max(axis=1)
    
    # Filter by threshold
    if thresh_lags is not None:
        joint_filtered = joint_data[joint_data[thresh_lags].max(axis=1) > thresh]
    else:
        joint_filtered = joint_data[joint_data['max_joint'] > thresh]
    
    return joint_filtered[['subject', 'electrode', 'max_joint']]

# Get thresholded electrodes
lags_for_thresh = np.arange(-500, 501, 50).tolist()
comp_thresh = get_max_joint_for_threshold(comp_data, thresh_joint, thresh_lags=lags_for_thresh)
prod_thresh = get_max_joint_for_threshold(prod_data, thresh_joint, thresh_lags=lags_for_thresh)

# remove bad electrodes (visual inspection)
comp_thresh = comp_thresh[~((comp_thresh['subject'] == '798') & 
                            (comp_thresh['electrode'].isin(['G1', 'G65'])))]
prod_thresh = prod_thresh[~((prod_thresh['subject'] == '798') & 
                            (prod_thresh['electrode'] == 'G104'))]


print(f"Comprehension: {len(comp_thresh)} electrodes above threshold")
print(f"Production: {len(prod_thresh)} electrodes above threshold")

Comprehension: 160 electrodes above threshold
Production: 232 electrodes above threshold


In [12]:
# Load electrode coordinates
from tfsplt_brainmap import read_coor

coords_dir = "/scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/data/plotting/brainplot/"
subjects = ["625", "676", "717", "798"]

print("Loading electrode coordinates...")
df_coor = read_coor(coords_dir, subjects)
df_coor.loc[df_coor['subject'] == '717', 'subject'] = '7170'
df_coor = df_coor.rename(columns={"name": "electrode", "MNI_X": "x", "MNI_Y": "y", "MNI_Z": "z"})
df_coor['subject'] = df_coor['subject'].astype(str)

print(f"Loaded coordinates for {len(df_coor)} electrodes")
print(f"Coordinate columns: {df_coor.columns.tolist()}")



print("\nPreparing comprehension data...")
comp_future = pu.prepare_video_data(comp_data, comp_thresh, 'sentence', lags_to_plot, df_coor)
comp_past = pu.prepare_video_data(comp_data, comp_thresh, 'sentence2', lags_to_plot, df_coor)
comp_word = pu.prepare_video_data(comp_data, comp_thresh, 'word', lags_to_plot, df_coor)


print("\nPreparing production data...")
prod_future = pu.prepare_video_data(prod_data, prod_thresh, 'sentence', lags_to_plot, df_coor)
prod_past =   pu.prepare_video_data(prod_data, prod_thresh, 'sentence2', lags_to_plot, df_coor)
prod_word =   pu.prepare_video_data(prod_data, prod_thresh, 'word', lags_to_plot, df_coor)

comp_future_ratio = pu.prepare_ratio_video_data(comp_data, comp_thresh, 'sentence', 'joint', lags_to_plot, df_coor, joint_max=True)
comp_past_ratio =   pu.prepare_ratio_video_data(comp_data, comp_thresh, 'sentence2', 'joint', lags_to_plot, df_coor, joint_max=True)
comp_word_ratio =   pu.prepare_ratio_video_data(comp_data, comp_thresh, 'word', 'joint', lags_to_plot, df_coor, joint_max=True)
prod_future_ratio = pu.prepare_ratio_video_data(prod_data, prod_thresh, 'sentence', 'joint', lags_to_plot, df_coor, joint_max=True)
prod_past_ratio =   pu.prepare_ratio_video_data(prod_data, prod_thresh, 'sentence2', 'joint', lags_to_plot, df_coor, joint_max=True)
prod_word_ratio =   pu.prepare_ratio_video_data(prod_data, prod_thresh, 'word', 'joint', lags_to_plot, df_coor, joint_max=True)

Loading electrode coordinates...
Loaded coordinates for 666 electrodes
Coordinate columns: ['electrode', 'name_NYUcoor', 'brain_data', 'coordinates', 'T1_X', 'T1_Y', 'T1_Z', 'x', 'y', 'z', 'type', 'reg1', 'reg1_percent', 'reg2', 'reg2_percent', 'reg3', 'reg3_percent', 'reg4', 'reg4_percent', 'subject']

Preparing comprehension data...
  sentence: 158 electrodes, 41 time points
  sentence2: 158 electrodes, 41 time points
  word: 158 electrodes, 41 time points

Preparing production data...
  sentence: 228 electrodes, 41 time points
  sentence2: 228 electrodes, 41 time points
  word: 228 electrodes, 41 time points
  sentence/joint: 158 electrodes, 41 time points
  sentence2/joint: 158 electrodes, 41 time points
  word/joint: 158 electrodes, 41 time points
  sentence/joint: 228 electrodes, 41 time points
  sentence2/joint: 228 electrodes, 41 time points
  word/joint: 228 electrodes, 41 time points


In [13]:
# Create custom colormaps
# Future: white -> green (for negative to positive correlations)
future_colors = ['white', 'lightgreen', 'green', 'darkgreen']
future_cmap = LinearSegmentedColormap.from_list('future', future_colors, N=256)

# Past: white -> red (for negative to positive correlations)
past_colors = ['white', 'lightcoral', 'red', 'darkred']
past_cmap = LinearSegmentedColormap.from_list('past', past_colors, N=256)

# Word: white -> orange
word_colors = ['white',  'orange', 'darkorange']
word_cmap = LinearSegmentedColormap.from_list('word', word_colors, N=256)

# Create output directory if it doesn't exist
import os
os.makedirs(output_dir, exist_ok=True)

# Get lag columns (excluding metadata)
lag_columns = [col for col in comp_future.columns if isinstance(col, (int, float, np.integer, np.floating))]

print(f"Will create videos with {len(lag_columns)} frames")
print(f"Lag range: {lag_columns[0]} to {lag_columns[-1]} ms")

Will create videos with 41 frames
Lag range: -1000 to 1000 ms


## Interactive Plotly Visualization

Create an interactive scrollable 3D brain visualization with separate rows for future, word, and past activity.

In [14]:
# Prepare comprehension data (word data needs to be loaded)
print("=" * 80)
print("Plotting comprehension timecourses...")
print("=" * 80)

comp_html_fig = pu.create_interactive_brain_viz_html(
    comp_future, 
    comp_word, 
    comp_past, 
    lags_to_plot,
    output_path=f"{output_dir}/comprehension_all_bands_interactive.html",
    title_prefix="Comprehension",
    vmin=0,
    vmax=0.25
)

print("=" * 80)
print("Plotting production timecourses...")
print("=" * 80)
prod_html_fig = pu.create_interactive_brain_viz_html(
    prod_future, 
    prod_word, 
    prod_past, 
    lags_to_plot,
    output_path=f"{output_dir}/production_all_bands_interactive.html",
    title_prefix="Production",
    vmin=0,
    vmax=0.25
)

print("=" * 80)
print("Plotting comprehension ratio timecourses...")
print("=" * 80)
comp_ratio_html_fig = pu.create_interactive_brain_viz_html(
    comp_future_ratio, 
    comp_word_ratio, 
    comp_past_ratio, 
    lags_to_plot,
    output_path=f"{output_dir}/comprehension_all_bands_ratio_interactive.html",
    title_prefix="Comprehension Ratio",
    vmin=0,
    vmax=0.8
)

print("=" * 80)
print("Plotting production ratio timecourses...")
print("=" * 80)
prod_ratio_html_fig = pu.create_interactive_brain_viz_html(
    prod_future_ratio, 
    prod_word_ratio, 
    prod_past_ratio, 
    lags_to_plot,
    output_path=f"{output_dir}/production_all_bands_ratio_interactive.html",
    title_prefix="Production Ratio",
    vmin=0,
    vmax=0.8
)


Plotting comprehension timecourses...
Creating interactive HTML visualization with glass brains: Comprehension
  Rendering 41 frames...
  Rendering frame 1/41
  Rendering frame 6/41
  Rendering frame 11/41
  Rendering frame 16/41
  Rendering frame 21/41
  Rendering frame 26/41
  Rendering frame 31/41
  Rendering frame 36/41
  Rendering frame 41/41
  Creating interactive Plotly figure...
  ✓ Saved interactive HTML to: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/roi_figures/comprehension_all_bands_interactive.html

Plotting production timecourses...
Creating interactive HTML visualization with glass brains: Production
  Rendering 41 frames...
  Rendering frame 1/41
  Rendering frame 6/41
  Rendering frame 11/41
  Rendering frame 16/41
  Rendering frame 21/41
  Rendering frame 26/41
  Rendering frame 31/41
  Rendering frame 36/41
  Rendering frame 41/41
  Creating interactive Plotly figure...
  ✓ Saved interactive HTML to: /scratch/gpfs/HASSON/ij9216/projects/code/2

## Generate Videos

### Video 1: Comprehension Future

In [ ]:
anim_comp_future = pu.create_video(
    df=comp_future,
    lags=lag_columns,
    cmap=future_cmap,
    title="Comprehension - Future Context",
    output_path=f"{output_dir}/comp_future_video.mp4",
    vmin=0.05,
    vmax=0.2,
    fps=3
)

Creating video: Comprehension - Future Context
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/roi_figures/comp_future_video.mp4
  Video saved!

  Video saved!



### Video 2: Comprehension Past

In [ ]:
anim_comp_past = create_video(
    df=comp_past,
    lags=lag_columns[::-1],  # Reverse order: 1000 to -1000
    cmap=past_cmap,
    title="Comprehension - Past Context",
    output_path=f"{output_dir}/comp_past_video.mp4",
    vmin=0.05,
    vmax=0.2,
    fps=3
)

Creating video: Comprehension - Past Context
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/roi_figures/comp_past_video.mp4
  Video saved!

  Video saved!



### Video 3: Production Future

In [ ]:
anim_prod_future = create_video(
    df=prod_future,
    lags=lag_columns,
    cmap=future_cmap,
    title="Production - Future Context",
    output_path=f"{output_dir}/prod_future_video.mp4",
    vmin=0.05,
    vmax=0.2,
    fps=3
)

Creating video: Production - Future Context
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/roi_figures/prod_future_video.mp4
  Video saved!

  Video saved!



### Video 4: Production Past

In [ ]:
anim_prod_past = create_video(
    df=prod_past,
    lags=lag_columns[::-1],  # Reverse order: 1000 to -1000
    cmap=past_cmap,
    title="Production - Past Context",
    output_path=f"{output_dir}/prod_past_video.mp4",
    vmin=0.05,
    vmax=0.2,
    fps=3
)

Creating video: Production - Past Context
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/roi_figures/prod_past_video.mp4
  Video saved!

  Video saved!



## Ratio Videos

### Ratio Video 1 - production past/joint ratio

In [ ]:
anim_prod_past_ratio = create_video(
    df=prod_past_ratio,
    lags=lag_columns,  # Reverse order: 1000 to -1000
    cmap=past_cmap,
    title="Production - Past/Joint Ratio",
    output_path=f"{output_dir}/prod_past_ratio_video.mp4",
    vmin=0,
    vmax=0.8,
    fps=3
)

Creating video: Production - Past/Joint Ratio
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/roi_figures/prod_past_ratio_video.mp4
  Video saved!

  Video saved!



### Ratio Video 2: Production Future/Joint ratio

In [ ]:
anim_prod_future_ratio = create_video(
    df=prod_future_ratio,
    lags=lag_columns,
    cmap=future_cmap,
    title="Production - Future/Joint Ratio",
    output_path=f"{output_dir}/prod_future_ratio_video.mp4",
    vmin=0,
    vmax=0.8,
    fps=3
)

Creating video: Production - Future/Joint Ratio
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/roi_figures/prod_future_ratio_video.mp4
  Video saved!

  Video saved!



### Ratio Video 3: Production Word/Joint ratio

In [ ]:
anim_prod_word_ratio = create_video(
    df=prod_word_ratio,
    lags=lag_columns,
    cmap=word_cmap,
    title="Production - Word/Joint Ratio",
    output_path=f"{output_dir}/prod_word_ratio_video.mp4",
    vmin=0,
    vmax=0.8,
    fps=3
)

Creating video: Production - Word/Joint Ratio
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/roi_figures/prod_word_ratio_video.mp4
  Video saved!

  Video saved!



### Ratio Video 3: Comprehension Past/Joint ratio

In [ ]:
anim_comp_past_ratio = create_video(
    df=comp_past_ratio,
    lags=lag_columns[::-1],  # Reverse order: 1000 to -1000
    cmap=past_cmap,
    title="Comprehension - Past/Joint Ratio",
    output_path=f"{output_dir}/comp_past_ratio_video.mp4",
    vmin=0,
    vmax=0.8,
    fps=3
)

Creating video: Comprehension - Past/Joint Ratio
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/roi_figures/comp_past_ratio_video.mp4
  Video saved!

  Video saved!



### Ratio Video 4: Comprehension Future/Joint ratio

In [ ]:
anim_comp_future_ratio = create_video(
    df=comp_future_ratio,
    lags=lag_columns,
    cmap=future_cmap,
    title="Comprehension - Future/Joint Ratio",
    output_path=f"{output_dir}/comp_future_ratio_video.mp4",
    vmin=0,
    vmax=0.8,
    fps=3
)

Creating video: Comprehension - Future/Joint Ratio
  Frames: 41
  Output: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/roi_figures/comp_future_ratio_video.mp4
  Video saved!

  Video saved!



In [ ]:
# def create_combined_video(df_list, lags, cmap_list, titles, output_path, vmin=0, vmax=0.8, fps=3):
#     """
#     Create a combined video with 3 rows, each showing a different ratio video.
    
#     Parameters:
#     - df_list: List of dataframes for the 3 rows.
#     - lags: List of lag columns to iterate over.
#     - cmap_list: List of colormaps for each row.
#     - titles: List of titles for each row.
#     - output_path: Path to save the output video.
#     - vmin, vmax: Color scale limits.
#     - fps: Frames per second for the video.
#     """
#     print(f"Creating combined video: {output_path}")
#     print(f"  Frames: {len(lags)}")
    
#     # Get coordinates (same for all rows)
#     coords_list = [df[['x', 'y', 'z']].values for df in df_list]
    
#     # Set up the figure and subplots
#     fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 12), constrained_layout=True)
    
#     def update_frame(frame_idx):
#         """Update function for animation."""
#         lag = lags[frame_idx]
        
#         for row, ax in enumerate(axes):
#             # Clear the axis completely to avoid clutter
#             ax.cla()
            
#             # Plot the markers for the current lag
#             values = df_list[row][lag].values
#             plotting.plot_markers(
#                 node_values=values,
#                 node_coords=coords_list[row],
#                 node_size=50,
#                 node_cmap=cmap_list[row],
#                 node_vmin=vmin,
#                 node_vmax=vmax,
#                 display_mode='lzry',
#                 colorbar=False,
#                 axes=ax
#             )
#             ax.set_title(f"{titles[row]} - Lag: {lag} ms", fontsize=12)
        
#         # Set the overall title for the figure
#         fig.suptitle("Combined Ratio Video", fontsize=16, y=0.95)
#         return fig,
    
#     # Create animation
#     anim = animation.FuncAnimation(
#         fig, 
#         update_frame, 
#         frames=len(lags),
#         interval=1000/fps,  # milliseconds per frame
#         blit=False
#     )
    
#     # Save video
#     writer = animation.FFMpegWriter(fps=fps, bitrate=1800)
#     anim.save(output_path, writer=writer)
    
#     plt.close(fig)
#     print(f"  Combined video saved at {output_path}\n")
#     return anim

# # Example usage:
# combined_video = create_combined_video(
#     df_list=[prod_past_ratio, prod_future_ratio, prod_word_ratio],
#     lags=lag_columns,
#     cmap_list=[past_cmap, future_cmap, word_cmap],
#     titles=["Production - Past/Joint Ratio", "Production - Future/Joint Ratio", "Production - Word/Joint Ratio"],
#     output_path=f"{output_dir}/combined_ratio_video.mp4",
#     vmin=0,
#     vmax=0.8,
#     fps=3
# )

Creating combined video: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/roi_figures/combined_ratio_video.mp4
  Frames: 41
  Combined video saved at /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/roi_figures/combined_ratio_video.mp4

  Combined video saved at /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/roi_figures/combined_ratio_video.mp4

